In [1]:
import requests
from pathlib import Path

PDB_ID = "7R73"

url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
pdb_text = requests.get(url).text

Path(f"{PDB_ID}.pdb").write_text(pdb_text)

print("Downloaded:", PDB_ID)
print("File size:", len(pdb_text), "characters")

Downloaded: 7R73
File size: 885978 characters


In [2]:
from collections import defaultdict

chains = defaultdict(set)

with open("7R73.pdb") as f:
    for line in f:
        if line.startswith("ATOM"):
            chain = line[21]
            resnum = line[22:26].strip()
            resname = line[17:20].strip()

            chains[chain].add((resnum, resname))

for chain, residues in chains.items():
    print(
        "Chain:", chain,
        "| residues:", len(residues)
    )

Chain: G | residues: 287
Chain: A | residues: 126


In [3]:
def extract_chain(input_pdb, output_pdb, chain_id):
    with open(input_pdb) as fin, open(output_pdb, "w") as fout:

        for line in fin:

            if line.startswith(("ATOM", "TER")):

                if len(line) > 21 and line[21] == chain_id:
                    fout.write(line)

        fout.write("END\n")


extract_chain(
    "7R73.pdb",
    "7R73_nanobody_experimental.pdb",
    "A"
)

extract_chain(
    "7R73.pdb",
    "7R73_antigen_gp120.pdb",
    "G"
)

print("Done")

Done


In [4]:
def count_atoms(filename):
    atoms = 0

    with open(filename) as f:
        for line in f:
            if line.startswith("ATOM"):
                atoms += 1

    return atoms


print(
    "Experimental nanobody atoms:",
    count_atoms("7R73_nanobody_experimental.pdb")
)

print(
    "Antigen atoms:",
    count_atoms("7R73_antigen_gp120.pdb")
)

Experimental nanobody atoms: 1877
Antigen atoms: 4466


In [5]:
import os

for f in [
    "7R73.pdb",
    "7R73_nanobody_experimental.pdb",
    "7R73_antigen_gp120.pdb"
]:
    print(f, os.path.getsize(f), "bytes")

7R73.pdb 885978 bytes
7R73_nanobody_experimental.pdb 152122 bytes
7R73_antigen_gp120.pdb 361831 bytes


In [6]:
import os

TARGET = "7R73A"

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if TARGET.lower() in file.lower():
            matches.append(os.path.join(root, file))

print("Found:")

for m in matches:
    print(m)

Found:
/kaggle/input/datasets/mennanabiil/targets-vhh/igfold_results/predictions/7R73A_igfold.fasta
/kaggle/input/datasets/mennanabiil/targets-vhh/igfold_results/predictions/7R73A_igfold.pdb


In [7]:
import os
import json
import shutil

TARGET = "7R73A"

# --------------------------------------------------
# 1. Find targets_vhh.json and IgFold prediction
# --------------------------------------------------

targets_path = None
igfold_source = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        full = os.path.join(root, file)

        if file == "targets_vhh.json":
            targets_path = full

        if file.lower() == "7r73a_igfold.pdb":
            igfold_source = full

print("targets:", targets_path)
print("IgFold :", igfold_source)

assert targets_path is not None, "targets_vhh.json not found"
assert igfold_source is not None, "7R73A IgFold prediction not found"


# --------------------------------------------------
# 2. Get the 7R73A sequence
# --------------------------------------------------

with open(targets_path) as f:
    targets = json.load(f)

target = None

for x in targets:
    tid = x["pdb"] + x["H"]

    if tid.upper() == TARGET:
        target = x
        break

assert target is not None, f"{TARGET} not found"

sequence = target["hseq"]

print("\nTarget:", TARGET)
print("Sequence length:", len(sequence))
print("CDR-H3:", target["CDRH3"])
print("CDR-H3 length:", len(target["CDRH3"]))


# --------------------------------------------------
# 3. Save FASTA
# --------------------------------------------------

fasta_path = "/kaggle/working/7R73A.fasta"

with open(fasta_path, "w") as f:
    f.write(">7R73A\n")
    f.write(sequence + "\n")

print("\nSaved FASTA:", fasta_path)


# --------------------------------------------------
# 4. Copy existing IgFold prediction
# --------------------------------------------------

output_igfold = "/kaggle/working/7R73A_IgFold.pdb"

shutil.copy(
    igfold_source,
    output_igfold
)

print("Copied IgFold prediction:", output_igfold)


# --------------------------------------------------
# 5. Quick check
# --------------------------------------------------

def count_atoms(filename):
    with open(filename) as f:
        return sum(
            line.startswith("ATOM")
            for line in f
        )

print("\nIgFold atoms:", count_atoms(output_igfold))
print(
    "IgFold file size:",
    os.path.getsize(output_igfold),
    "bytes"
)

targets: /kaggle/input/datasets/mennanabiil/targets-vhh/targets_vhh.json
IgFold : /kaggle/input/datasets/mennanabiil/targets-vhh/igfold_results/predictions/7R73A_igfold.pdb

Target: 7R73A
Sequence length: 126
CDR-H3: AAKWRPLRYSDNPSNSDYNY
CDR-H3 length: 20

Saved FASTA: /kaggle/working/7R73A.fasta
Copied IgFold prediction: /kaggle/working/7R73A_IgFold.pdb

IgFold atoms: 618
IgFold file size: 50147 bytes


In [8]:
!apt-get update -qq
!apt-get install -y -qq hmmer

!pip install -q biopython openmm
!pip install -q git+https://github.com/openmm/pdbfixer.git

!rm -rf /kaggle/working/ANARCI
!git clone -q https://github.com/oxpig/ANARCI.git /kaggle/working/ANARCI

%cd /kaggle/working/ANARCI
!python setup.py install
%cd /kaggle/working

!pip install -q ImmuneBuilder

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64.deb ...
Unpacking hmmer (3.3.2+dfsg-1) ...
Setting up libdivsufsort3:amd64 (2.0.1-5) ...
Setting up hmmer (3.3.2+dfsg-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/s

In [9]:
import anarci
import pdbfixer
import openmm

from ImmuneBuilder import NanoBodyBuilder2

print("ANARCI ✓")
print("PDBFixer ✓")
print("OpenMM ✓", openmm.__version__)
print("NanoBodyBuilder2 ✓")

ANARCI ✓
PDBFixer ✓
OpenMM ✓ 8.5.2
NanoBodyBuilder2 ✓


In [10]:
from ImmuneBuilder import NanoBodyBuilder2
import os

predictor = NanoBodyBuilder2()

prediction = predictor.predict({
    "H": sequence
})

nbb2_output = "/kaggle/working/7R73A_NBB2.pdb"

prediction.save(nbb2_output)

print("Saved:", nbb2_output)
print("Atoms:", count_atoms(nbb2_output))
print("Size:", os.path.getsize(nbb2_output), "bytes")

Saved: /kaggle/working/7R73A_NBB2.pdb
Atoms: 1894
Size: 153641 bytes


In [11]:
# Clean previous copy if there is one
!rm -rf /kaggle/working/ml-simplefold

# Same fork + branch used in our benchmark
!git clone -q \
    --branch fix-multichain-fasta-parsing \
    https://github.com/omar-A-hassan/ml-simplefold.git \
    /kaggle/working/ml-simplefold

# Install SimpleFold
%cd /kaggle/working/ml-simplefold
!pip install -q -e .

# Install ESM dependency
!pip install -q git+https://github.com/facebookresearch/esm.git

%cd /kaggle/working

print("Installation finished")

/kaggle/working/ml-simplefold
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.5/392.5 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 7.3 MB/s eta 0:00:00
 

In [12]:
import shutil

print("SimpleFold:", shutil.which("simplefold"))

SimpleFold: /usr/local/bin/simplefold


In [13]:
from pathlib import Path

inference_file = Path(
    "/kaggle/working/ml-simplefold/src/simplefold/inference.py"
)

text = inference_file.read_text()

old = "esm_model = esm_model.to(device)"
new = "esm_model = esm_model.half().to(device)"

if old in text:
    text = text.replace(old, new, 1)
    inference_file.write_text(text)
    print("ESM FP16 patch applied ✓")

elif new in text:
    print("ESM FP16 patch already applied ✓")

else:
    print("Could not find expected line")

ESM FP16 patch applied ✓


In [14]:
!grep -n "esm_model = esm_model" \
/kaggle/working/ml-simplefold/src/simplefold/inference.py

188:        esm_model = esm_model.half().to(device)
196:        esm_model = esm_model_mlx


In [15]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [16]:
import os
import shutil
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nSimpleFold:", shutil.which("simplefold"))

files_to_check = [
    "/kaggle/working/7R73A.fasta",
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
]

for f in files_to_check:
    print(os.path.basename(f), "✓" if os.path.exists(f) else "MISSING")

CUDA: True
GPU: Tesla T4

SimpleFold: /usr/local/bin/simplefold
7R73A.fasta ✓
7R73A_IgFold.pdb ✓
7R73A_NBB2.pdb ✓


In [17]:
from pathlib import Path

inference_file = Path(
    "/kaggle/working/ml-simplefold/src/simplefold/inference.py"
)

text = inference_file.read_text()

old = "esm_model = esm_model.to(device)"
new = "esm_model = esm_model.half().to(device)"

if new in text:
    print("Patch already applied ✓")
elif old in text:
    text = text.replace(old, new, 1)
    inference_file.write_text(text)
    print("Patch applied ✓")
else:
    print("Expected line not found")

Patch already applied ✓


In [18]:
!grep -n "esm_model = esm_model" /kaggle/working/ml-simplefold/src/simplefold/inference.py

188:        esm_model = esm_model.half().to(device)
196:        esm_model = esm_model_mlx


In [19]:
!rm -rf /kaggle/working/simplefold_7R73A

!simplefold \
    --simplefold_model simplefold_100M \
    --num_steps 500 \
    --tau 0.01 \
    --nsample_per_protein 1 \
    --seed 12345 \
    --fasta_path /kaggle/working/7R73A.fasta \
    --output_dir /kaggle/working/simplefold_7R73A \
    --output_format pdb \
    --backend torch

MLX not installed, skip importing MLX related packages.
Running protein folding with SimpleFold ...
Seed set to 12345
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  368M  100  368M    0     0  14.4M      0  0:00:25  0:00:25 --:--:-- 14.7M
Folding model simplefold_100M loaded.
Using device: cuda.
Downloading: "https://github.com/facebookresearch/esm/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t36_3B_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t36_3B_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t36_3B_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t36_3B_UR50D-contact-regression.pt
pLM ESM-3B loaded with torch backend.
Checking input data.
Found 1 examples to process.
Processing input data.
100%|██████████████████████████████████████████

In [20]:
import os

found = []

for root, dirs, files in os.walk("/kaggle/working/simplefold_7R73A"):
    for file in files:
        if file.endswith(".pdb"):
            path = os.path.join(root, file)
            found.append(path)
            print(path)

print("\nTotal PDB files found:", len(found))

/kaggle/working/simplefold_7R73A/predictions_simplefold_100M/7R73A_sampled_0.pdb

Total PDB files found: 1


In [21]:
import shutil
import os

simplefold_source = found[0]
simplefold_output = "/kaggle/working/7R73A_SimpleFold.pdb"

shutil.copy(simplefold_source, simplefold_output)

def count_atoms(filename):
    with open(filename) as f:
        return sum(line.startswith("ATOM") for line in f)

print("Saved:", simplefold_output)
print("Atoms:", count_atoms(simplefold_output))
print("Size:", os.path.getsize(simplefold_output), "bytes")

Saved: /kaggle/working/7R73A_SimpleFold.pdb
Atoms: 972
Size: 78893 bytes


In [22]:
import os

for f in [
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_SimpleFold.pdb",
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_nanobody_experimental.pdb",
]:
    print(os.path.basename(f), "✓" if os.path.exists(f) else "MISSING")

7R73A_IgFold.pdb ✓
7R73A_NBB2.pdb ✓
7R73A_SimpleFold.pdb ✓
7R73_antigen_gp120.pdb ✓
7R73_nanobody_experimental.pdb ✓


In [23]:
import os

def pdb_summary(path):
    atoms = 0
    residues = set()
    chains = set()

    with open(path) as f:
        for line in f:
            if line.startswith("ATOM"):
                atoms += 1

                chain = line[21].strip() or "_"
                resnum = line[22:26].strip()
                icode = line[26].strip()

                chains.add(chain)
                residues.add((chain, resnum, icode))

    return {
        "file": os.path.basename(path),
        "atoms": atoms,
        "residues": len(residues),
        "chains": sorted(chains)
    }


files = [
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_nanobody_experimental.pdb",
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_SimpleFold.pdb",
]

for f in files:
    print(pdb_summary(f))

{'file': '7R73_antigen_gp120.pdb', 'atoms': 4466, 'residues': 287, 'chains': ['G']}
{'file': '7R73_nanobody_experimental.pdb', 'atoms': 1877, 'residues': 126, 'chains': ['A']}
{'file': '7R73A_IgFold.pdb', 'atoms': 618, 'residues': 126, 'chains': ['H']}
{'file': '7R73A_NBB2.pdb', 'atoms': 1894, 'residues': 126, 'chains': ['H']}
{'file': '7R73A_SimpleFold.pdb', 'atoms': 972, 'residues': 126, 'chains': ['A']}


In [24]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from pathlib import Path
import os


def strip_hydrogens_and_set_chain(input_pdb, output_pdb, new_chain):
    """
    Keep protein ATOM records only,
    remove hydrogen atoms,
    and assign one consistent chain ID.
    """
    
    with open(input_pdb) as fin, open(output_pdb, "w") as fout:
        for line in fin:

            if not line.startswith("ATOM"):
                continue

            atom_name = line[12:16].strip()
            element = line[76:78].strip().upper()

            # Detect hydrogen
            is_hydrogen = (
                element == "H"
                or (
                    not element
                    and atom_name.lstrip("0123456789").startswith("H")
                )
            )

            if is_hydrogen:
                continue

            # Change chain ID
            line = line[:21] + new_chain + line[22:]

            fout.write(line)

        fout.write("END\n")


def prepare_structure(input_pdb, output_pdb, chain_id):

    temp = "/kaggle/working/_temp_noH.pdb"

    strip_hydrogens_and_set_chain(
        input_pdb,
        temp,
        chain_id
    )

    fixer = PDBFixer(filename=temp)

    # Detect missing residues but DO NOT build them
    fixer.findMissingResidues()
    fixer.missingResidues = {}

    # Add only missing atoms in residues already present
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()

    with open(output_pdb, "w") as f:
        PDBFile.writeFile(
            fixer.topology,
            fixer.positions,
            f,
            keepIds=True
        )

    os.remove(temp)

    print("Prepared:", os.path.basename(output_pdb))

In [25]:
prepare_structure(
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_antigen_DOCK.pdb",
    "G"
)

prepare_structure(
    "/kaggle/working/7R73_nanobody_experimental.pdb",
    "/kaggle/working/7R73_experimental_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_IgFold_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_NBB2_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_SimpleFold.pdb",
    "/kaggle/working/7R73A_SimpleFold_DOCK.pdb",
    "H"
)

Prepared: 7R73_antigen_DOCK.pdb
Prepared: 7R73_experimental_DOCK.pdb
Prepared: 7R73A_IgFold_DOCK.pdb
Prepared: 7R73A_NBB2_DOCK.pdb
Prepared: 7R73A_SimpleFold_DOCK.pdb


In [26]:
dock_files = [
    "/kaggle/working/7R73_antigen_DOCK.pdb",
    "/kaggle/working/7R73_experimental_DOCK.pdb",
    "/kaggle/working/7R73A_IgFold_DOCK.pdb",
    "/kaggle/working/7R73A_NBB2_DOCK.pdb",
    "/kaggle/working/7R73A_SimpleFold_DOCK.pdb",
]

for f in dock_files:
    print(pdb_summary(f))

{'file': '7R73_antigen_DOCK.pdb', 'atoms': 2283, 'residues': 287, 'chains': ['G']}
{'file': '7R73_experimental_DOCK.pdb', 'atoms': 973, 'residues': 126, 'chains': ['H']}
{'file': '7R73A_IgFold_DOCK.pdb', 'atoms': 973, 'residues': 126, 'chains': ['H']}
{'file': '7R73A_NBB2_DOCK.pdb', 'atoms': 973, 'residues': 126, 'chains': ['H']}
{'file': '7R73A_SimpleFold_DOCK.pdb', 'atoms': 973, 'residues': 126, 'chains': ['H']}


In [27]:
import os
import shutil
from pathlib import Path

backup_dir = Path("/kaggle/working/7R73_docking_project")
backup_dir.mkdir(exist_ok=True)

important_files = [
    # Original complex
    "7R73.pdb",
    
    # Original separated structures
    "7R73_antigen_gp120.pdb",
    "7R73_nanobody_experimental.pdb",
    
    # Sequence
    "7R73A.fasta",
    
    # Raw model predictions
    "7R73A_IgFold.pdb",
    "7R73A_NBB2.pdb",
    "7R73A_SimpleFold.pdb",
    
    # Standardized docking inputs
    "7R73_antigen_DOCK.pdb",
    "7R73_experimental_DOCK.pdb",
    "7R73A_IgFold_DOCK.pdb",
    "7R73A_NBB2_DOCK.pdb",
    "7R73A_SimpleFold_DOCK.pdb",
]

for name in important_files:
    src = Path("/kaggle/working") / name
    
    if src.exists():
        shutil.copy2(src, backup_dir / name)
        print("✓", name)
    else:
        print("MISSING:", name)

# Create ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/7R73_docking_project_backup",
    "zip",
    backup_dir
)

print("\nBackup created:")
print(zip_path)
print("Size:", round(os.path.getsize(zip_path)/1024/1024, 2), "MB")

✓ 7R73.pdb
✓ 7R73_antigen_gp120.pdb
✓ 7R73_nanobody_experimental.pdb
✓ 7R73A.fasta
✓ 7R73A_IgFold.pdb
✓ 7R73A_NBB2.pdb
✓ 7R73A_SimpleFold.pdb
✓ 7R73_antigen_DOCK.pdb
✓ 7R73_experimental_DOCK.pdb
✓ 7R73A_IgFold_DOCK.pdb
✓ 7R73A_NBB2_DOCK.pdb
✓ 7R73A_SimpleFold_DOCK.pdb

Backup created:
/kaggle/working/7R73_docking_project_backup.zip
Size: 0.5 MB
